In [20]:
"""
    Fahion MNIST verisetini kullanarak moda ürün tasarımı gerçekleştirmek.
        
"""

'\n    Fahion MNIST verisetini kullanarak moda ürün tasarımı gerçekleştirmek.\n        \n'

In [21]:
import tensorflow as tf
from tensorflow.keras import layers
import numpy as np
import matplotlib.pyplot as plt
import os 
from tensorflow.keras.datasets import fashion_mnist

In [22]:
BUFFER_SIZE = 60000
BATCH_SIZE = 128
NOISE_DIM = 100

(train_data,_),(_,_) = fashion_mnist.load_data()
train_data = train_data.reshape(-1,28,28,1).astype('float32')
train_data = (train_data - 127.5) / 127.5
train_dataset = tf.data.Dataset.from_tensor_slices(train_data).shuffle(BUFFER_SIZE).batch(BATCH_SIZE)

In [23]:
def make_generator_model():
    model = tf.keras.Sequential([
        layers.Dense(7*7*256, use_bias=False, input_shape=(NOISE_DIM,)),
        layers.BatchNormalization(),
        layers.LeakyReLU(),
        layers.Reshape((7, 7, 256)),
        
        layers.Conv2DTranspose(128, (5, 5), strides=(1, 1), padding='same', use_bias=False),
        layers.BatchNormalization(),
        layers.LeakyReLU(),
        
        layers.Conv2DTranspose(64, (5, 5), strides=(2, 2), padding='same', use_bias=False),
        layers.BatchNormalization(),
        layers.LeakyReLU(),
        
        
        layers.Conv2DTranspose(1, (5, 5), strides=(2,2), padding='same', use_bias=False, activation='tanh')
    ])
    return model

In [24]:
def make_discriminator_model():
    model = tf.keras.Sequential([
        layers.Conv2D(64, (5, 5), strides=(2, 2), padding='same', input_shape=[28, 28, 1]),
        layers.LeakyReLU(),
        layers.Dropout(0.3),
        
        layers.Conv2D(128, (5, 5), strides=(2, 2), padding='same'),
        layers.LeakyReLU(),
        layers.Dropout(0.3),
        
        layers.Flatten(),
        layers.Dense(1)
    ])
    return model

In [25]:
cross_entropy = tf.keras.losses.BinaryCrossentropy()

def discriminator_loss(real_output, fake_output):
    real_loss = cross_entropy(tf.ones_like(real_output), real_output)
    fake_loss = cross_entropy(tf.zeros_like(fake_output), fake_output)
    total_loss = real_loss + fake_loss
    return total_loss

def generator_loss(fake_output):
    return cross_entropy(tf.ones_like(fake_output), fake_output)

generator = make_generator_model()
discriminator = make_discriminator_model()

generator_optimizer = tf.keras.optimizers.Adam(1e-4)
discriminator_optimizer = tf.keras.optimizers.Adam(1e-4)

In [26]:
seed = tf.random.normal([16, NOISE_DIM])

In [27]:
def generate_and_save_images(model, epoch, test_input):
    predictions = model(test_input, training=False)
    fig = plt.figure(figsize=(4,4))
    
    for i in range(predictions.shape[0]):
        plt.subplot(4, 4, i+1)
        plt.imshow(predictions[i, :, :, 0] * 127.5 + 127.5, cmap='gray')
        plt.axis('off')
    
    if not os.path.exists('images'):
        os.makedirs('images')
    plt.savefig('images/image_at_epoch_{:04d}.png'.format(epoch))
    plt.close()

In [28]:
def train(dataset,epochs):
    for epoch in range(1,epochs+1):
        gen_loss_total = 0
        disc_loss_total = 0
        batch_count = 0
        
        for image_batch in dataset:
            noise = tf.random.normal([BATCH_SIZE, NOISE_DIM])
            with tf.GradientTape() as gen_tape, tf.GradientTape() as disc_tape:
                generated_images = generator(noise, training=True)
                
                real_output = discriminator(image_batch, training=True)
                fake_output = discriminator(generated_images, training=True)
                
                gen_loss = generator_loss(fake_output)
                disc_loss = discriminator_loss(real_output, fake_output)
            
            gradients_of_generator = gen_tape.gradient(gen_loss, generator.trainable_variables)
            gradients_of_discriminator = disc_tape.gradient(disc_loss, discriminator.trainable_variables)
            
            generator_optimizer.apply_gradients(zip(gradients_of_generator, generator.trainable_variables))
            discriminator_optimizer.apply_gradients(zip(gradients_of_discriminator, discriminator.trainable_variables))
            
            gen_loss_total += gen_loss
            disc_loss_total += disc_loss
            
            batch_count += 1  
        
        print(f'Epoch {epoch}, Gen Loss: {gen_loss_total/batch_count}, Disc Loss: {disc_loss_total/batch_count}')
        generate_and_save_images(generator, epoch, seed)

train(train_dataset, 50)
    

Epoch 1, Gen Loss: 7.497324466705322, Disc Loss: 0.7508655786514282
Epoch 2, Gen Loss: 1.9533616304397583, Disc Loss: 0.9702490568161011
Epoch 3, Gen Loss: 1.9032433032989502, Disc Loss: 1.0922698974609375
Epoch 4, Gen Loss: 1.2731603384017944, Disc Loss: 1.0683430433273315
Epoch 5, Gen Loss: 1.8485294580459595, Disc Loss: 0.979010283946991


KeyboardInterrupt: 